In [2]:
# 1 - LOGISTIC REGRESSION 
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score, classification_report
import matplotlib.pyplot as plt
import csv


import warnings
warnings.filterwarnings("ignore")

# Robust CSV load (auto-detect separator)
df = pd.read_csv("processed_cardio_dataset.csv", sep=None, engine='python')
# Basic clean
if 'id' in df.columns: df = df.drop('id', axis=1)
# convert boolean columns to ints (if any)
df = df.replace({True:1, False:0})

print("Columns:", df.columns.tolist())
label_col = 'cardio'

# Prepare data
X = df.drop(label_col, axis=1)
y = df[label_col]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# Baseline
base = LogisticRegression(max_iter=2000, random_state=42)
base.fit(X_train_s, y_train)

# Grid-search tuning
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
param_grid = {'C':[0.01,0.1,1,10], 'solver':['liblinear','lbfgs']}
grid = GridSearchCV(LogisticRegression(max_iter=2000, random_state=42), param_grid, cv=cv, scoring='f1', n_jobs=-1)
grid.fit(X_train_s, y_train)
best = grid.best_estimator_

# Evaluate
def eval_model(name, model, Xs, ytrue):
    ypred = model.predict(Xs)
    try:
        yscore = model.predict_proba(Xs)[:,1]
        auc = roc_auc_score(ytrue, yscore)
    except Exception:
        auc = np.nan
    print(f"\n{name} results:")
    print("Best params (if grid):", getattr(model, 'get_params', lambda: None)())
    print("Accuracy:", accuracy_score(ytrue, ypred))
    print("Precision:", precision_score(ytrue, ypred, zero_division=0))
    print("Recall:", recall_score(ytrue, ypred, zero_division=0))
    print("F1:", f1_score(ytrue, ypred, zero_division=0))
    print("AUC:", auc)
    print("Confusion matrix:\n", confusion_matrix(ytrue, ypred))
    print(classification_report(ytrue, ypred))

eval_model("Logistic (baseline)", base, X_test_s, y_test)
eval_model("Logistic (GridSearch best)", best, X_test_s, y_test)

# Short conclusion comment:
# - Metrics: Accuracy/F1/Precision/Recall/AUC chosen because this is binary medical classification.
# - Validation: Stratified k-fold (GridSearchCV). Focus on F1 or Recall if missing positives is costly.


Columns: ['age', 'height', 'weight', 'ap_hi', 'ap_lo', 'smoke', 'alco', 'active', 'cardio', 'age_years', 'BMI', 'gender_Male', 'cholesterol_Normal', 'cholesterol_Well Above Normal', 'gluc_Normal', 'gluc_Well Above Normal']

Logistic (baseline) results:
Best params (if grid): {'C': 1.0, 'class_weight': None, 'dual': False, 'fit_intercept': True, 'intercept_scaling': 1, 'l1_ratio': None, 'max_iter': 2000, 'multi_class': 'deprecated', 'n_jobs': None, 'penalty': 'l2', 'random_state': 42, 'solver': 'lbfgs', 'tol': 0.0001, 'verbose': 0, 'warm_start': False}
Accuracy: 0.7282181818181818
Precision: 0.7593845113290497
Recall: 0.6599559147685525
F1: 0.7061875933642582
AUC: 0.792313776396786
Confusion matrix:
 [[5522 1423]
 [2314 4491]]
              precision    recall  f1-score   support

           0       0.70      0.80      0.75      6945
           1       0.76      0.66      0.71      6805

    accuracy                           0.73     13750
   macro avg       0.73      0.73      0.73   